# Week 6 · Day 4 — Non-parametric tests, multiple comparisons and power

A permutation test by hand, Mann–Whitney vs Welch, Fisher vs chi-square, Holm by hand, and the power calculation for Jasur's SMS test.

In [1]:
from pathlib import Path
import hashlib
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
DATA = Path("../..")           # this notebook lives in data/python/week6/, so "../.." is the data/ folder
AS_OF = pd.Timestamp("2026-10-01")
con = sqlite3.connect(DATA / "raw" / "nur.db")
rng = np.random.default_rng(2026)   # one seeded generator: the same "random" numbers on every run


def check(value, fingerprints, dp=0):
    """Right or wrong without showing the answer: the rounded value's fingerprint must be one of the expected ones."""
    fp = hashlib.sha256(f"nur-w6:{round(float(value), dp):.{dp}f}".encode()).hexdigest()[:12]
    assert fp in fingerprints, "not yet: this result is not the expected one — check your function and run again"

In [2]:
# The three frames of the week (built exactly as in week 5; run, do not edit)
loans = pd.read_sql("""
    SELECT l.loan_id, l.disbursed_at, l.principal, l.term_months, l.risk_grade, l.status,
           p.code, p.name AS product, a.channel, a.credit_score, b.branch_name
    FROM loans l JOIN products p USING (product_id) JOIN applications a USING (application_id)
    JOIN branches b ON b.branch_id = l.branch_id""", con, parse_dates=["disbursed_at"])
inst = pd.read_sql("SELECT * FROM installments", con, parse_dates=["due_date"])
pay = (pd.read_sql("SELECT * FROM payments", con, parse_dates=["paid_at"])
         .sort_values("payment_id").drop_duplicates(["installment_id", "paid_at", "amount"]))   # 52 850 clean payments

# f: one row per loan whose FIRST instalment fell due on or before as-of - 30 days; fpd30 = 1 if not fully paid within 30 days
first = inst[(inst["installment_no"] == 1) & (inst["due_date"] <= AS_OF - pd.Timedelta(days=30))]
paid1 = pay[pay["paid_at"] <= AS_OF].groupby("installment_id", as_index=False).agg(paid=("amount", "sum"), first_paid=("paid_at", "min"))
f = first.merge(paid1, on="installment_id", how="left").fillna({"paid": 0})
f["fpd30"] = ((f["paid"] < f["total_due"]) | ((f["first_paid"] - f["due_date"]).dt.days > 30)).astype(int)
f = f.merge(loans, on="loan_id", validate="1:1")

# x: one row per clean payment, with its due date, days late and the loan's grade / product
x = pay.merge(inst[["installment_id", "due_date"]], on="installment_id").merge(
        loans[["loan_id", "risk_grade", "code", "channel"]], on="loan_id")
x["days_late"] = (x["paid_at"].dt.normalize() - x["due_date"]).dt.days
x["late"] = (x["paid_at"] > x["due_date"]).astype(int)
assert len(loans) == 7828 and len(f) == 7099 and f["fpd30"].sum() == 91 and len(x) == 52850
print(f"loans {len(loans)} · FPD30 {f.fpd30.sum()} / {len(f)} = {f.fpd30.mean():.2%} · clean payments {len(x)}")

loans 7828 · FPD30 91 / 7099 = 1.28% · clean payments 52850


## 1. `perm_test(y, is_a, B=10_000, seed=2026)` — difference in means of y between group A and the rest
Shuffle the labels B times; p = share of shuffled |differences| ≥ the observed |difference|. Return `(observed_diff, p)`.

In [3]:
def perm_test(y, is_a, B=10_000, seed=2026):
    """(mean(y[is_a]) - mean(y[~is_a]), two-sided permutation p-value)."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [4]:
ab = f[f.channel.isin(["app", "branch"])]
obs, p = perm_test(ab["fpd30"].to_numpy(), (ab["channel"] == "app").to_numpy())
check(obs, ['0798f63fc221', '3073fe7454d1', 'e02052882e96'], 6)
check(p, ['028b8ac7829d', '028dab6338b4', '0841966cd4c2', '0884282fe700', '0c324676a0c1', '0c85702a52b1', '0dc42e0376d8', '0e630fd580e3', '0f591185388e', '0f68635dd7b3', '0fd0b0e2c62e', '10aee5b12b62', '121ab67966e6', '181130669fc8', '1bab0548c848', '1c981dabf0d0', '1f4e6dd60235', '202b32d088dd', '20941477cb22', '21e0be13c0e2', '236f9d578d53', '239d6ae8c3be', '2599a9766330', '260124c7d81f', '2793638ece57', '27c918fac1c7', '2a9ea3f95bfb', '2ad06467157a', '31194297e3cc', '319b88ff039a', '372f2fd23818', '3884330c0a53', '390eba67ddff', '39f2dc49fb4a', '3c9b0cbca331', '3f6fa608a63f', '3fc47d0ca5d0', '40aace45de42', '41790ad9ab19', '42e010d84109', '4487a73bc449', '45052835751a', '46ee7a907177', '47c50f0776bc', '4862524f485f', '4892c1cffdac', '48a939952f07', '4d82ab0dab4b', '4de4613e5d68', '534c19abdd48', '57029e177126', '59ce3a28aaaf', '5a8bcd450f8e', '5c2ecfa2fa91', '5cc14573dc4d', '5d5a9c5017c0', '5da3da42165a', '5dd4e21d5560', '5de7fbfd57af', '65d3b25eeaa8', '6649a0a6ead9', '67e430622c79', '6ab6f02a5ef3', '6d1e5ab266d4', '6d7483bce4ad', '6f176e57303a', '717a6d2dc41d', '752a35ba8ec3', '779c993f5f05', '7932c81e586d', '7b9d35167eab', '7ef896b566ff', '7f5a8546be2c', '80a5fc26a666', '80f463aafb4a', '811960d98d34', '81773ec33918', '826a02dc75bb', '828ab757f455', '847057c9c254', '894f979bb59c', '899da74bf417', '89da610f52fe', '8e25fb2afce4', '8ef11f9cf3b8', '91162ab494af', '9155de6c4670', '921e871c1a2f', '972c8c087a26', '9752d8ed19dd', '97ccf93051da', '9cb506ca3d31', '9db17978be64', '9dfd55b65b46', '9e63ecedd688', '9f07936639ce', '9f8f38249b86', 'a0bf163fe20d', 'a29f7dcd3491', 'a5d30e8c954f', 'a6b28bd3b271', 'a9cd7920402e', 'ac61ca80ddac', 'ae998f54d0dd', 'b1c8b9ba8de5', 'b3a0d17fe8b6', 'b433da16caca', 'b43c28dd4092', 'b4c5e43c61c2', 'b78e0789a2f4', 'b817c68d761f', 'b9309c0f4f0a', 'b9ed9749a9c8', 'ba1022793a35', 'baf26ed40c26', 'bb228ffff432', 'bd0948233d98', 'c3b52680e8b7', 'c5381d824239', 'c994a4e8b893', 'c9bcf36f6d25', 'ca473e7d43e2', 'cd4dfd5ae861', 'ce1a30db917c', 'd0ba3929e800', 'd273e98a3dfd', 'd6019790742f', 'd6e8786f455f', 'd8024c6760a2', 'd850b62b7848', 'da62a2117917', 'dc7149a01630', 'dd57c953274e', 'dfff147c5c85', 'e082ddb552ab', 'e14ac4981acd', 'e1d6611c2a51', 'e2be0df658bb', 'e35ea67386f6', 'e49a1fe2391b', 'e611b1d818b2', 'e8b4ec7239cc', 'ed0e89daa8d8', 'ed57ccd70a59', 'f16d44c8074f', 'f1f02562b4f9', 'f3c0cc67a087', 'f8ddb0882fc0', 'faf6e6c8293f', 'ff7d0dde3ae9', 'ffbdce74d240'], 4)              # close to the z-test's p
print(f"perm_test OK: diff {obs:.4%}, p = {p:.4f}")

NotImplementedError: TODO: write me, then run the assert cell below

## 2. Mann–Whitney vs Welch — days late, Auto vs SME (skewed: most pay 1–3 days early, a few 30+ late)

In [5]:
a, b = x.loc[x.code == "AUTO", "days_late"], x.loc[x.code == "SME", "days_late"]
print(pd.DataFrame({"AUTO": a.describe(), "SME": b.describe()}).round(2).T)
print("share paid late:", round((a > 0).mean(), 4), round((b > 0).mean(), 4))
print("Welch:       ", stats.ttest_ind(a, b, equal_var=False))
print("Mann-Whitney:", stats.mannwhitneyu(a, b, alternative="two-sided"))
print("Kruskal-Wallis, 4 grades:", stats.kruskal(*[g["days_late"] for _, g in x.groupby("risk_grade")]))

       count  mean   std  min  25%  50%  75%   max
AUTO  4688.0 -0.21  5.56 -3.0 -2.0 -1.0  0.0  60.0
SME   8222.0  0.04  6.18 -3.0 -2.0 -1.0  0.0  60.0
share paid late: 0.1009 0.1241
Welch:        TtestResult(statistic=np.float64(-2.335815775966022), pvalue=np.float64(0.019519368801419416), df=np.float64(10613.48609645537))
Mann-Whitney: MannwhitneyuResult(statistic=np.float64(19075370.5), pvalue=np.float64(0.3226498515883295))
Kruskal-Wallis, 4 grades: KruskalResult(statistic=np.float64(934.9563813553733), pvalue=np.float64(2.315265924213046e-202))


## 3. Fisher vs chi-square on a small table — the 2026-05 vintage, partner vs branch

In [6]:
v = f[f.disbursed_at.dt.to_period("M") == "2026-05"].groupby("channel")["fpd30"].agg(["sum", "count"])
print(v)
t = np.array([[v.loc["partner", "sum"], v.loc["partner", "count"] - v.loc["partner", "sum"]],
              [v.loc["branch", "sum"],  v.loc["branch", "count"] - v.loc["branch", "sum"]]])
chi2, p, dof, expected = stats.chi2_contingency(t)
print("expected:", expected.round(2).tolist(), "→ two cells below 5")
print(f"chi-square (Yates) p = {p:.4f} · without correction p = {stats.chi2_contingency(t, correction=False)[1]:.4f} · Fisher p = {stats.fisher_exact(t)[1]:.4f}")

         sum  count
channel            
app        1    131
branch     2    179
partner    4     67
expected: [[1.63, 65.37], [4.37, 174.63]] → two cells below 5
chi-square (Yates) p = 0.0832 · without correction p = 0.0281 · Fisher p = 0.0485


## 4. Ten branches, ten tests — `holm(pvals)` by hand

In [7]:
rows = []
for name, g in f.groupby("branch_name"):
    k, n = g["fpd30"].sum(), len(g)
    kr, nr = f["fpd30"].sum() - k, len(f) - n
    rows.append([name, k, n, round(100 * k / n, 2), stats.fisher_exact([[k, n - k], [kr, nr - kr]])[1]])
br = pd.DataFrame(rows, columns=["branch", "fpd", "n", "rate_pct", "p"]).sort_values("p").reset_index(drop=True)
br

,branch,fpd,n,rate_pct,p
0,Bukhara Central,14,586,2.39,0.019906
1,Tashkent North,8,921,0.87,0.273270
2,Nukus Central,3,440,0.68,0.376904
3,Tashkent Central,14,913,1.53,0.432614
4,Tashkent South,14,917,1.53,0.434507
5,Andijan Central,6,635,0.94,0.577908
6,Fergana Central,9,834,1.08,0.742536
7,Namangan Central,8,634,1.26,1.000000
8,Samarkand Central,10,782,1.28,1.000000
9,Termez Central,5,437,1.14,1.000000


In [8]:
def holm(pvals, alpha=0.05):
    """Holm-adjusted p-values, in the input order: sort ascending, multiply the i-th smallest (i = 0..m-1) by (m - i),
    take the running maximum, cap at 1."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [9]:
from statsmodels.stats.multitest import multipletests
assert np.allclose(holm(br["p"].to_numpy()), multipletests(br["p"], method="holm")[1])
h3 = holm([0.04, 0.01, 0.03])
check(h3[0], ['26d44d21c9e0'], 6); check(h3[1], ['ba5431ae9d35'], 6); check(h3[2], ['26d44d21c9e0'], 6)
br["holm"] = holm(br["p"].to_numpy()); br["bonferroni"] = np.minimum(br["p"] * len(br), 1)
print("holm OK"); br.round(4)

NotImplementedError: TODO: write me, then run the assert cell below

In [10]:
# The problem, simulated: 20 000 imaginary books in which all ten branches have the SAME true FPD rate.
# Each branch is tested against that rate with an exact two-sided binomial test at alpha = 0.05.
p0, ns = 91 / 7099, br["n"].to_numpy()
S = 20_000
k = rng.binomial(ns, p0, size=(S, len(ns)))
pv = np.minimum(1, 2 * np.minimum(stats.binom.cdf(k, ns, p0), stats.binom.sf(k - 1, ns, p0)))
print(f"single test, share 'significant': {np.mean(pv < 0.05):.1%} (an exact test on small counts stays under 5 %)")
print(f"books with at least one 'significant' branch: {np.mean((pv < 0.05).any(axis=1)):.1%} (formula 1 - 0.95^10 = {1 - 0.95**10:.1%})")
print(f"… after Bonferroni (p < 0.005): {np.mean((pv < 0.005).any(axis=1)):.1%}")

single test, share 'significant': 3.6% (an exact test on small counts stays under 5 %)
books with at least one 'significant' branch: 30.6% (formula 1 - 0.95^10 = 40.1%)
… after Bonferroni (p < 0.005): 2.3%


## 5. Power and sample size — `n_per_arm`, `achieved_power`, `mde_rate`
Use `statsmodels`: `proportion_effectsize(p1, p2)` (Cohen's h) and `NormalIndPower` / `zt_ind_solve_power`; two-sided, alpha 0.05.

In [11]:
def n_per_arm(p1, p2, power=0.80, alpha=0.05):
    """Loans needed in EACH group to detect p1 vs p2 (two-sided), rounded UP to an int."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [12]:
def achieved_power(p1, p2, n, alpha=0.05):
    """Power of a two-sided test with n loans in each group."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [13]:
def mde_rate(p1, n, power=0.80, alpha=0.05):
    """The lowest p2 < p1 that n loans per group detect with the given power (solve h(p1, p2) = required h)."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [14]:
check(n_per_arm(0.0128, 0.010), ['237dcca7321d'])
check(achieved_power(0.0128, 0.010, 1000), ['5cc39fdbacea'], 3)
check(achieved_power(0.0128, 0.010, 2000), ['376e763d605c'], 3)
check(mde_rate(0.0128, 2000), ['1b3bbac112d6', '3afcc1d44ee8', '44d600535ea4', '51998d1ca22f', '5eca7463152b', '61a88a54e355', '682963c2259a', '8f6dbb1fc4a5', '9133a5abb2e9', '94deba440ff6', '95badc1ee254', '9644c3a317b5', '9e1a88682b75', 'a5a6808c145d', 'b51dd23b826e', 'c5292f0b5081', 'd68e91554705', 'db59853380d1', 'e2d7939144f9', 'ec2a878b18e3', 'fd2a67d5cbab'], 5)
print("power functions OK")

NotImplementedError: TODO: write me, then run the assert cell below

In [15]:
# Power by simulation, to trust the formula: 2 000 loans per arm, true rates 1.28 % vs 1.00 %
S = 20_000
ka, kb = rng.binomial(2000, 0.0128, S), rng.binomial(2000, 0.010, S)
pool = (ka + kb) / 4000
with np.errstate(invalid="ignore", divide="ignore"):
    z = (ka - kb) / 2000 / np.sqrt(pool * (1 - pool) * 2 / 2000)
print(f"simulated power at 2 000 per arm: {np.mean(2 * stats.norm.sf(np.abs(np.nan_to_num(z))) < 0.05):.3f}")

simulated power at 2 000 per arm: 0.125


In [16]:
# The A/A idea: split the real FPD loans into two random halves of 2 000 that got the SAME treatment; how often "significant"?
yy = f["fpd30"].to_numpy()
hits = 0
for _ in range(2000):
    idx = rng.permutation(len(yy))
    a_, b_ = yy[idx[:2000]], yy[idx[2000:4000]]
    pool = (a_.sum() + b_.sum()) / 4000
    se = np.sqrt(pool * (1 - pool) * 2 / 2000)
    hits += 2 * stats.norm.sf(abs(a_.mean() - b_.mean()) / se) < 0.05
print(f"A/A false-positive rate: {hits / 2000:.1%} (should be about 5 %)")

A/A false-positive rate: 4.9% (should be about 5 %)
